# Notebook 7: H, O, T Nodes Insertion

This notebook integrates all generated **H**, **O**, and **T** nodes into the G2 graph, using the previously detected structural communities.

---

### 1. K-means Clustering of S and A Embeddings  
We apply the **K-means** algorithm to the embeddings of all **S** and **A** nodes to form  
**k = √(number of S and A nodes)** clusters.

For each **H** node, we then identify the **top 5 closest semantic clusters** based on centroid similarity.

---

### 2. Linking S/A Nodes to H Nodes  
Within each community:

If an **S** or **A** node belongs to one of the **top 5 semantic clusters** associated with an **H** node, that node is linked to the corresponding **H** node.

---

### 3. O Nodes Insertion  
We insert all **O** nodes (High-level Overview nodes) and link each **O** node to the **H** node from which its summary was produced.

The updated graph is saved as **G3 — community augmented graph**.

---

### 4. T Nodes Insertion  
We insert all **T** nodes (original text chunks) and connect each one to its child **S** nodes using the `source_id` attribute.  
This preserves the original textual meaning that may have been abstracted away during LLM-based semantic decomposition.

The final graph is saved as **G4 — text inserted graph**.


In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\graphs
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pandas as pd
#H nodes
medical_summary = pd.read_parquet("data/nodes/community/medical_communities_summary.parquet")
medical_summary

,node_id,community_summary
0,medical-H-17,Here are distinct categories of high-level inf...
1,medical-H-11,Here are five distinct categories of high-leve...
2,medical-H-20,Here are the distinct categories of high-level...
3,medical-H-7,Here are distinct categories of high-level inf...
4,medical-H-27,Here are distinct categories of high-level inf...
5,medical-H-1,Here are distinct categories of high-level inf...
6,medical-H-23,Here are distinct categories of high-level inf...
7,medical-H-22,Here are distinct categories of high-level inf...
8,medical-H-4,Here are distinct categories of high-level inf...
9,medical-H-6,Here are distinct categories of high-level inf...


In [4]:
#O nodes
medical_overview = pd.read_parquet("data/nodes/community/medical_communities_overview.parquet")
medical_overview

,node_id,community_overview
0,medical-H-17-O-0,"Skin Cancer: Basal Cell Carcinoma, Squamous Ce..."
1,medical-H-11-O-0,"Head and Neck Cancer: Anatomy, Diagnosis, Trea..."
2,medical-H-20-O-0,"Cancer Management: Surgical Treatment, Staging..."
3,medical-H-7-O-0,"Lymphatic System, Cancer Metastasis, Breast Ca..."
4,medical-H-27-O-0,"NCCN Guidelines: Cancer Patient Empowerment, P..."
5,medical-H-1-O-0,"Cancer Genetics, Mutations, Genomic Testing, a..."
6,medical-H-23-O-0,"Cancer: Risk Factors, Pathogenesis, Staging, S..."
7,medical-H-22-O-0,"HPV-Related Cancers, Biomarkers, Treatment Sid..."
8,medical-H-4-O-0,"Patient Health Management, Advocacy, Support"
9,medical-H-6-O-0,"Cancer Care: Diagnostics, Personalized Treatme..."


In [5]:
import pickle
#communities
with open(f"{root_path}/graphs/data//nodes/community/medical_communities.pkl", "rb") as f:
    communities_medical = pickle.load(f)
communities_medical

defaultdict(list,
            {17: ['medical-0-S-0',
              'medical-0-S-0-R-0',
              'medical-0-S-0-R-1',
              'medical-0-S-0-R-2',
              'medical-0-S-0-R-3',
              'medical-0-S-0-R-4',
              'medical-0-S-1',
              'medical-0-S-1-R-0',
              'medical-0-S-1-R-1',
              'medical-0-S-1-R-2',
              'medical-0-S-1-R-3',
              'medical-0-S-1-R-4',
              'medical-0-S-1-R-5',
              'medical-0-S-1-R-6',
              'medical-0-S-1-R-7',
              'medical-0-S-1-R-8',
              'medical-0-S-1-R-9',
              'medical-0-S-1-R-10',
              'medical-0-S-1-R-11',
              'medical-0-S-1-R-12',
              'medical-0-S-2-R-1',
              'medical-0-S-2-R-2',
              'medical-0-S-3',
              'medical-0-S-3-R-0',
              'medical-0-S-3-R-1',
              'medical-0-S-3-R-2',
              'medical-0-S-3-R-3',
              'medical-0-S-3-R-4',
       

In [6]:
#T nodes
medical_chunks = pd.read_parquet(f"{root_path}/chunking/medical_chunks.parquet")
medical_chunks['node_id'] = "medical-" + medical_chunks.index.astype(str)
medical_chunks[["node_id","chunk"]]

,node_id,chunk
0,medical-0,About basal cell skin cancer What is basal cel...
1,medical-1,Hypodermis – The deepest tissue layer is made ...
2,medical-2,Causes and risk factors Basal cell skin cancer...
3,medical-3,Recurrence is when basal cell skin cancer retu...
4,medical-4,2 Testing for basal cell skin cancer 10 Genera...
...,...,...
549,medical-549,Is there a portal where I can get copies of my...
550,medical-550,"Is home care after treatment needed? If yes, w..."
551,medical-551,What will you target? What if I’ve already had...
552,medical-552,What are the chances the cancer will return or...


In [7]:
#S,N,R,A nodes
#G2 - graph
with open(f"{root_path}/graphs/data/graphs/G2_medical_attribute_enriched_graph.pkl", "rb") as f:
    medical_g2 = pickle.load(f)
medical_g2

{'medical-0-S-0': <graphs.Node.Node at 0x2087ec40910>,
 'medical-0-S-0-R-0': <graphs.Node.Node at 0x2087ec40e80>,
 'medical-0-S-0-R-1': <graphs.Node.Node at 0x2087ec40e50>,
 'medical-0-S-0-R-2': <graphs.Node.Node at 0x2081708b070>,
 'medical-0-S-0-R-3': <graphs.Node.Node at 0x2081708a3e0>,
 'medical-0-S-0-R-4': <graphs.Node.Node at 0x2081708b850>,
 'medical-0-S-1': <graphs.Node.Node at 0x2081708aa10>,
 'medical-0-S-1-R-0': <graphs.Node.Node at 0x2081708bbb0>,
 'medical-0-S-1-R-1': <graphs.Node.Node at 0x2081708abf0>,
 'medical-0-S-1-R-2': <graphs.Node.Node at 0x2081708b8b0>,
 'medical-0-S-1-R-3': <graphs.Node.Node at 0x2081708bcd0>,
 'medical-0-S-1-R-4': <graphs.Node.Node at 0x2081708b0d0>,
 'medical-0-S-1-R-5': <graphs.Node.Node at 0x2081708ae90>,
 'medical-0-S-1-R-6': <graphs.Node.Node at 0x2081708a980>,
 'medical-0-S-1-R-7': <graphs.Node.Node at 0x2081708b970>,
 'medical-0-S-1-R-8': <graphs.Node.Node at 0x2081708b9a0>,
 'medical-0-S-1-R-9': <graphs.Node.Node at 0x2081708bb80>,
 'med

In [8]:
import json
import faiss
import numpy as np
#embeddings of S,A,H,T nodes

#load faiss index file
index = faiss.read_index("data/embedding/medical_index.faiss")
with open("data/embedding/medical_ids.json", "r") as f:
    medical_embedding_ids = json.load(f)
index.ntotal, len(medical_embedding_ids)

#reconstruct vectors from faiss index
num_vectors = index.ntotal
dimension = index.d
embeddings = np.zeros((num_vectors, dimension), dtype='float32')
for i in range(num_vectors):
    embeddings[i] = index.reconstruct(i)

print("Embeddings shape:", embeddings.shape)
embeddings

Embeddings shape: (4426, 768)


array([[-0.06428347, -0.05439772, -0.04260492, ..., -0.0463121 ,
        -0.03141087,  0.0049261 ],
       [-0.04751787, -0.05777418, -0.02587977, ..., -0.04238609,
        -0.00359848,  0.0272001 ],
       [-0.06036756, -0.067439  , -0.05656344, ..., -0.06152818,
        -0.01679316,  0.01591949],
       ...,
       [ 0.04048249, -0.0287718 , -0.03389297, ..., -0.00295302,
         0.0417009 ,  0.00099732],
       [ 0.00630154, -0.0139234 , -0.01267361, ..., -0.09236696,
         0.01060561, -0.01778929],
       [ 0.01245843, -0.01854508, -0.01106029, ..., -0.08220403,
         0.01130273,  0.05532239]], dtype=float32)

In [9]:
import re
#separate T nodes, H nodes, S-A nodes
#id pattern
medical_T_id_pattern = re.compile(r"^medical-\d+$")
medical_H_id_pattern = re.compile(r"^medical-H-\d+$")
#id lists
medical_embedding_T_ids = []
medical_embedding_H_ids = []
medical_embedding_SA_ids = []
#index
medical_embedding_H_index = []
medical_embedding_SA_index = []


for i in range(num_vectors):
    current_id = medical_embedding_ids[i]
    if medical_T_id_pattern.match(current_id):
        medical_embedding_T_ids.append(current_id)
    elif medical_H_id_pattern.match(current_id):
        medical_embedding_H_ids.append(current_id)
        medical_embedding_H_index.append(i)
    else:
        medical_embedding_SA_ids.append(current_id)
        medical_embedding_SA_index.append(i)

H_embeddings = embeddings[medical_embedding_H_index]
SA_embeddings = embeddings[medical_embedding_SA_index]
SA_embeddings.shape, H_embeddings.shape

((3831, 768), (41, 768))

In [10]:
import math
#k means on embeddings
#hyperparameters
k = math.floor(math.sqrt(SA_embeddings.shape[0]))
niter = 1000

#init model and train
kmeans = faiss.Kmeans(d=index.d,k=k,niter=niter,verbose=True,spherical=True,gpu=False)
kmeans.cp.metric_type = faiss.METRIC_INNER_PRODUCT
kmeans.train(SA_embeddings)

#get centroids and assign
centroids = kmeans.centroids
_, SA_assignments = kmeans.index.search(SA_embeddings, 1)
SA_assignments = SA_assignments.reshape(-1)

#cluster->SA nodes mapping
clusters = {i: [] for i in range(k)}
for idx, cid in enumerate(SA_assignments):
    clusters[cid].append(medical_embedding_SA_ids[idx])

print("Number of clusters:",len(clusters))
clusters

Number of clusters: 61


{0: ['medical-115-S-1',
  'medical-115-S-2',
  'medical-172-S-4',
  'medical-172-S-5',
  'medical-172-S-7',
  'medical-228-S-0',
  'medical-249-S-7',
  'medical-250-S-1',
  'medical-348-S-2',
  'medical-348-S-3',
  'medical-348-S-4',
  'medical-372-S-1',
  'medical-380-S-4',
  'medical-380-S-5',
  'medical-381-S-1',
  'medical-553-S-4',
  'medical-553-S-7',
  'medical-N-1559-A-0',
  'medical-N-2126-A-0',
  'medical-N-2133-A-0',
  'medical-N-2134-A-0',
  'medical-N-2893-A-0',
  'medical-N-2894-A-0',
  'medical-N-2895-A-0',
  'medical-N-2898-A-0',
  'medical-N-3546-A-0',
  'medical-N-3861-A-0',
  'medical-N-4379-A-0',
  'medical-N-2889-A-0',
  'medical-N-3864-A-0'],
 1: ['medical-181-S-1',
  'medical-485-S-0',
  'medical-485-S-1',
  'medical-485-S-3',
  'medical-485-S-4',
  'medical-486-S-0',
  'medical-486-S-1',
  'medical-486-S-2',
  'medical-486-S-3',
  'medical-487-S-0',
  'medical-487-S-1',
  'medical-487-S-2',
  'medical-487-S-3',
  'medical-487-S-5',
  'medical-487-S-6',
  'medica

In [11]:
#H node->cluster mapping
#using top k to avoid disconnected H nodes
_, H_assignments = kmeans.index.search(H_embeddings, 5)
H_assignments, H_assignments.shape
H_node_clusters = {}
for i in range(H_assignments.shape[0]):
    H_node_clusters[medical_embedding_H_ids[i]] = H_assignments[i].tolist()

H_node_clusters

{'medical-H-17': [25, 57, 59, 19, 31],
 'medical-H-11': [31, 8, 19, 4, 58],
 'medical-H-20': [19, 60, 5, 4, 57],
 'medical-H-7': [4, 32, 19, 37, 49],
 'medical-H-27': [58, 19, 60, 5, 15],
 'medical-H-1': [13, 3, 19, 4, 48],
 'medical-H-23': [19, 31, 4, 37, 52],
 'medical-H-22': [19, 31, 4, 25, 30],
 'medical-H-4': [48, 19, 35, 58, 60],
 'medical-H-6': [19, 58, 4, 60, 43],
 'medical-H-5': [19, 18, 52, 4, 60],
 'medical-H-25': [29, 4, 53, 19, 48],
 'medical-H-36': [31, 37, 19, 9, 4],
 'medical-H-3': [51, 19, 28, 54, 4],
 'medical-H-26': [33, 12, 4, 19, 48],
 'medical-H-19': [11, 30, 19, 4, 31],
 'medical-H-21': [19, 48, 4, 60, 39],
 'medical-H-24': [52, 19, 37, 18, 4],
 'medical-H-14': [4, 21, 19, 47, 22],
 'medical-H-8': [19, 60, 58, 55, 45],
 'medical-H-15': [50, 14, 19, 4, 30],
 'medical-H-9': [4, 8, 49, 19, 48],
 'medical-H-28': [19, 60, 5, 53, 45],
 'medical-H-18': [19, 4, 47, 60, 54],
 'medical-H-12': [16, 17, 19, 40, 4],
 'medical-H-0': [56, 43, 4, 34, 19],
 'medical-H-30': [4, 19

In [12]:
#create and link H and O nodes
for H_id in medical_summary["node_id"].tolist():
    #ids
    O_id = f"{H_id}-O-0"
    #data
    H_content = medical_summary[medical_summary["node_id"] == H_id]["community_summary"].iloc[0]
    O_content = medical_overview[medical_overview["node_id"] == O_id]["community_overview"].iloc[0]
    #node creation
    H_node = Node(
        id = H_id,
        node_type = "H",
        source = "",
        content = H_content
    )

    O_node = Node(
        id = O_id,
        node_type = "O",
        source = "",
        content = O_content
    )

    #link H and O node
    H_node.link(O_node)
    O_node.link(H_node)

    #link H node with S,A nodes in the same community and cluster
    current_cluster = set()
    for cluster_id in H_node_clusters[H_id]:
        current_cluster |= set(clusters[cluster_id])
    current_community = set(communities_medical[int(re.search(r"medical-H-(\d+)", H_id).group(1))])
    relevant_nodes = current_cluster&current_community
    for node_id in relevant_nodes:
        if node_id == H_id:
            continue
        relevant_node = medical_g2[node_id]
        relevant_node.link(H_node)
        H_node.link(relevant_node)
    
    #add to node list
    medical_g2[H_id] = H_node
    medical_g2[O_id] = O_node



In [13]:
for node_id in list(medical_g2.keys()):
    node = medical_g2[node_id]
    if node.node_type == "H" and len(node.edges) <= 1:
        print(node_id)
        medical_g2.pop(node_id, None)


In [14]:
with open(f"{root_path}/graphs/data/graphs/G3_medical_community_augmented_graph.pkl", "wb") as f:
    pickle.dump(medical_g2, f)
with open(f"{root_path}/graphs/data/graphs/G3_medical_community_augmented_graph.pkl", "rb") as f:
    medical_g3 = pickle.load(f)

In [15]:
#create and link T nodes
for node_id in list(medical_g3.keys()):
    node = medical_g3[node_id]
    if node.node_type != "S":
        continue
    source_id = node.source
    if source_id not in medical_g3:
        T_node = Node(
            id = source_id,
            node_type = "T",
            source = "",
            content = medical_chunks[medical_chunks["node_id"] == source_id]["chunk"].iloc[0]
        )
        medical_g3[source_id] = T_node
    T_node = medical_g3[source_id]
    T_node.link(node)
    node.link(T_node)

In [16]:
with open(f"{root_path}/graphs/data/graphs/G4_text_inserted_graph.pkl", "wb") as f:
    pickle.dump(medical_g3, f)
with open(f"{root_path}/graphs/data/graphs/G4_text_inserted_graph.pkl", "rb") as f:
    medical_g4 = pickle.load(f)
medical_g4

{'medical-0-S-0': <graphs.Node.Node at 0x2081aa1be20>,
 'medical-0-S-0-R-0': <graphs.Node.Node at 0x2081708b4c0>,
 'medical-0-S-0-R-1': <graphs.Node.Node at 0x208782df370>,
 'medical-0-S-0-R-2': <graphs.Node.Node at 0x208782df3a0>,
 'medical-0-S-0-R-3': <graphs.Node.Node at 0x208782df3d0>,
 'medical-0-S-0-R-4': <graphs.Node.Node at 0x208782df400>,
 'medical-0-S-1': <graphs.Node.Node at 0x208782df430>,
 'medical-0-S-1-R-0': <graphs.Node.Node at 0x208782df460>,
 'medical-0-S-1-R-1': <graphs.Node.Node at 0x208782df490>,
 'medical-0-S-1-R-2': <graphs.Node.Node at 0x208782df4c0>,
 'medical-0-S-1-R-3': <graphs.Node.Node at 0x208782df4f0>,
 'medical-0-S-1-R-4': <graphs.Node.Node at 0x208782df520>,
 'medical-0-S-1-R-5': <graphs.Node.Node at 0x208782df550>,
 'medical-0-S-1-R-6': <graphs.Node.Node at 0x208782df580>,
 'medical-0-S-1-R-7': <graphs.Node.Node at 0x208782df5b0>,
 'medical-0-S-1-R-8': <graphs.Node.Node at 0x208782df5e0>,
 'medical-0-S-1-R-9': <graphs.Node.Node at 0x208782df610>,
 'med